In [ ]:
# Cell 1: Set paths and processing parameters.
from pathlib import Path
import sys

# Run from implement/ or its parent project directory.
IMPLEMENT_DIR = Path.cwd().resolve()
if not (IMPLEMENT_DIR / "crystal_pipeline.py").is_file():
    IMPLEMENT_DIR = IMPLEMENT_DIR / "implement"
if not (IMPLEMENT_DIR / "crystal_pipeline.py").is_file():
    raise FileNotFoundError("Start Jupyter from implement/ or its parent directory.")
sys.path.insert(0, str(IMPLEMENT_DIR))
PROJECT_ROOT = IMPLEMENT_DIR.parent
INPUT_DIR = IMPLEMENT_DIR / "data"
RUN_DIR = IMPLEMENT_DIR / "outputs" / "trial_archived_features_001"  # Use a new name for different settings.
COLOR_REFERENCE_PATH = INPUT_DIR / "models" / "rf_step1.pkl"
CONFIG = {
    "data_dir": "data/mag500",
    "target_plates": {"old": ["pl01", "pl02", "pl04"], "new": ["pl01", "pl02", "pl04"]},
    "images_per_plate": 3,
    "crop": {"top": 60, "bottom": 100, "left": 60, "right": 100},
    "n_segments": 300, "compactness": 10.0,
}
print(f"Project: {PROJECT_ROOT}\nRun: {RUN_DIR}")

In [ ]:
# Cell 2: Save selected image paths and settings (no image loading).
# Re-running Cells 2-4 overwrites outputs. Use a new run name for a separate experiment.
from crystal_pipeline import prepare_run

prepare_run(PROJECT_ROOT, RUN_DIR, CONFIG)

In [ ]:
# Cell 3: Crop images, compute SLIC segments, and save per-image Lab means.
from crystal_preprocessing import run_preprocessing

run_preprocessing(PROJECT_ROOT, RUN_DIR)

In [ ]:
# Cell 4: Save pixel-Lab corrections using the RF model's own reference.
from crystal_preprocessing import run_color_correction

run_color_correction(RUN_DIR, COLOR_REFERENCE_PATH)

In [ ]:
# Cell 5: Calculate and save the original 18 RF features, one image at a time.
from crystal_features import run_feature_extraction

run_feature_extraction(RUN_DIR)

In [ ]:
# Cell 6: Group superpixels by standardized features using PCA and GMM.
# Cluster IDs are local to each image; they are not crystal class labels.
from crystal_gmm import run_gmm

run_gmm(RUN_DIR, n_components=3, pca_components=15, probability_cut=0.90)

In [ ]:
# Cell 7: Classify GMM clusters and save crystal class area ratios.
from crystal_rf import run_rf_classification

model_dir = INPUT_DIR / "models"
stage2_paths = {"1": model_dir / "rf_step2_group2.pkl", "2": model_dir / "rf_step2_group3.pkl", "3": model_dir / "rf_step2_group4.pkl"}
# The original pipeline applies the group31 model to hierarchical label "20".
stage3_paths = {"20": model_dir / "rf_step3_group31.pkl"}
run_rf_classification(RUN_DIR, model_dir / "rf_step1.pkl", stage2_paths, stage3_paths)

In [ ]:
# Cell 8: Create the manual correction table; keep existing edits on rerun.
from crystal_review import prepare_corrections

CORRECTIONS_PATH = prepare_corrections(RUN_DIR)

In [ ]:
# Cell 9: Inspect the original image and its GMM clusters for one plate.
# Use image_id and cluster_id to find the matching row in cluster_corrections.csv.
from crystal_review import show_clusters

show_clusters(RUN_DIR, group="old", plate="pl01")

In [ ]:
# Cell 10: Apply nonblank new_class entries and recalculate areas.
# Original RF outputs are preserved; reruns overwrite rf_corrected/.
from crystal_corrections import apply_manual_corrections

apply_manual_corrections(RUN_DIR)

In [ ]:
# Cell 11: Fit subclass GMMs across images; save results without plotting.
# Parent labels include descendants (prefix matching), as in the original notebook.
from crystal_subclasses import run_subclass_gmm

SUBCLASS_CONFIGS = [
    {"name": "class200_GMM2", "n_components": {200: 2}},
    {"name": "class200_GMM3", "n_components": {200: 3}},
]
run_subclass_gmm(RUN_DIR, SUBCLASS_CONFIGS, pca_components=10)

In [ ]:
# Cell 12: Review saved subclass results without rerunning GMM.
# Each panel shows an entire original GMM cluster, not an individual subclass mask.
from crystal_subclasses import show_subclasses

show_subclasses(RUN_DIR, config_name="class200_GMM2", representative_n=20)

In [ ]:
# Cell 13: Apply a saved subclass configuration and recalculate area ratios.
# Labels become parent_subclass (e.g. 200_0); other classes stay unchanged.
from crystal_subclasses import apply_subclasses

SELECTED_SUBCLASS_CONFIG = "class200_GMM2"  # Change to class200_GMM3 to apply the three-way split.
apply_subclasses(RUN_DIR, config_name=SELECTED_SUBCLASS_CONFIG)

In [ ]:
# Cell 14: Sampling systematic error — histogram and stability plot.
from crystal_sampling import analyze_sampling

AREA_CSV = RUN_DIR / "subclass_gmm" / SELECTED_SUBCLASS_CONFIG / "applied" / "area_ratios.csv"
# Parent label 0 includes both 0 and any 0_* subclasses present in the results.
# Current data: 3 images per plate, so choose 2; set n_use=4 after processing 8 images.
analyze_sampling(AREA_CSV, target_labels=["0"], class_name="fine crystal", group="new", plate="pl02", n_use=2, max_error=None)

In [ ]:
# Cell 15: Plot four area categories vs photon yield in a single 2-by-2 figure.
# Bars show leave-one-out ranges for plates with at least two images.
from crystal_photon import plot_area_photon

CLASS_GROUPS = {
    "fine": ["0"], "elongated": ["10", "11", "12"],
    "plate-like": ["200", "201", "21"], "empty": ["30", "31", "32"],
}
AREA_CSV = RUN_DIR / "subclass_gmm" / SELECTED_SUBCLASS_CONFIG / "applied" / "area_ratios.csv"
plot_area_photon(AREA_CSV, INPUT_DIR / "tables" / "photon_yield.csv", CLASS_GROUPS, min_area=0.0)

In [ ]:
# Cell 16: Fit the bounded weighted photon-yield model for the four area categories.
# Fit scores use training plates; plate-omission results are saved separately.
from crystal_photon_model import fit_photon_model

PLATE_CSV = AREA_CSV.parent / "photon_yield" / "plate_area_photon.csv"
fit_photon_model(PLATE_CSV, categories=list(CLASS_GROUPS), bounds=(0, 500))

In [ ]:
# Cell 17: Fit the photon-yield model with empty area multiplied by coverage.
# Use cluster-based coverage means; missing plates retain their uncorrected empty area.
# Coverage variability is saved but not included in model sensitivity bars.
from crystal_coverage import fit_coverage_model

fit_coverage_model(PLATE_CSV, INPUT_DIR / "tables" / "coverage_settings.csv", INPUT_DIR / "tables" / "coverage_result.csv", categories=list(CLASS_GROUPS), bounds=(0, 500))